# Phase 8 — eksperimen counting B v2 dan video review

Notebook **CPU** ini memakai cache pengukuran Phase 8 yang sudah selesai, kemudian menjalankan **kebijakan counting B v2 baru** dan membuat video anotasi terpisah. YOLO, BoT-SORT, dan encoder ReID **tidak dijalankan ulang**. Hasil/model/cache B v1 lama tetap utuh.

**Input Kaggle:** tambahkan output notebook `chrisbiran/phase-8-tgdi-full-video-research` serta dataset `chrisbiran/tgdi-traffic-videos`. Manifest donor harus tepat mempunyai SHA `79e8cf19747c63b34f8389d8f525925fd0c69e966f3d12ffb89716a5c89bd0dc`, dari run `20261015T083731_664817Z`. Pilih accelerator **None/CPU**. Tidak ada instalasi model atau akses token GitHub/Kaggle dari notebook ini.

Geometri baru memakai unggahan pengguna TGDI-WEEKDAY-PAGI pada koordinat native 640×480 sebagai template bagi delapan video sesuai instruksi pengguna. Transfer template merupakan keputusan eksperimen; keandalannya belum terbukti. Policy dan geometri terikat SHA dari GitHub/bootstrap source.

Default **Run All** memproses seluruh delapan video. Gunakan `VIDEO_IDS = ['weekday_pagi']` untuk uji subset. Pengujian satu video tidak dinyatakan sebagai penyelesaian delapan video.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path(globals().get('REPO_ROOT', Path.cwd())).resolve()
if not (REPO_ROOT / 'notebooks').is_dir() and (REPO_ROOT.parent / 'notebooks').is_dir():
    REPO_ROOT = REPO_ROOT.parent
INPUT_ROOT = globals().get('INPUT_ROOT', '/kaggle/input')
SOURCE_RUN_ROOT = globals().get('SOURCE_RUN_ROOT', None)  # None: temukan donor dengan SHA manifest yang tepat
SOURCE_VIDEO_ROOT = globals().get('SOURCE_VIDEO_ROOT', None)  # None: cari sumber aktual lalu verifikasi SHA
OUTPUT_ROOT = globals().get('OUTPUT_ROOT', '/kaggle/working/results/phase_8_b_v2')
VIDEO_IDS = globals().get('VIDEO_IDS', None)  # None = seluruh delapan; contoh ['weekday_pagi']
RENDER = globals().get('RENDER', True)  # False hanya membuat hasil replay counting; belum membuat video
RUN_ID = globals().get('RUN_ID', None)  # None = identitas run baru; hasil lama tidak ditimpa

sys.path.insert(0, str(REPO_ROOT / 'notebooks'))
print({'repo_root': str(REPO_ROOT), 'output_root': str(OUTPUT_ROOT),
       'video_ids': VIDEO_IDS if VIDEO_IDS is not None else 'ALL_EIGHT', 'render': RENDER})


## Verifikasi environment dan kontrak software

OpenCV/NumPy dipakai untuk membaca/render frame; FFmpeg/FFprobe untuk MP4 dan pemeriksaan media. Tiga suite baru harus benar-benar memuat tes, lulus, dan **tidak mempunyai skipped test**. Fixture sintetis memeriksa kontrak software, bukan akurasi counting pada video nyata. Tidak ada dependency GPU/model yang diimpor.


In [ ]:
import platform
import shutil
import unittest
import cv2
import numpy as np
import phase_8_b_v2_counting_tests
import phase_8_b_v2_annotation_tests
import phase_8_b_v2_batch_tests

for tool in ('ffmpeg', 'ffprobe'):
    if shutil.which(tool) is None:
        raise RuntimeError(f'{tool} tidak tersedia; hentikan sebelum replay/render')
print({'python': platform.python_version(), 'opencv': cv2.__version__, 'numpy': np.__version__,
       'execution': 'CPU_RETAINED_MEASUREMENT_REPLAY_NEW_B_V2_COUNTING_NO_MODEL_RERUN'})
test_modules = (phase_8_b_v2_counting_tests, phase_8_b_v2_annotation_tests, phase_8_b_v2_batch_tests)
software_results = []
for module in test_modules:
    suite = unittest.defaultTestLoader.loadTestsFromModule(module)
    if suite.countTestCases() == 0:
        raise RuntimeError('Suite kosong: ' + module.__name__)
    result = unittest.TextTestRunner(verbosity=1).run(suite)
    if not result.wasSuccessful() or result.skipped:
        raise RuntimeError('Software guard gagal atau ada tes dilewati: ' + module.__name__)
    software_results.append({'module': module.__name__, 'tests_run': result.testsRun, 'skipped': len(result.skipped)})
print({'software_suites': software_results, 'tests_are_model_benchmark': False})


## Replay kebijakan baru dan render

Controller memverifikasi cache asli, snapshot hasil B v1, sumber video, relasi retained, geometri lama, geometri baru, serta policy sebelum menghitung ulang kebijakan B v2. Semua input tetap read-only; hasil B v2 menggunakan namespace dan direktori baru. Kegagalan sebagian harus dilaporkan sebagai kegagalan, bukan hasil delapan video selesai.

Tiga jalur bukti dibedakan dalam ledger dan panel: **LINE_MARGIN** (crossing terukur), **OCCLUSION_DIFF_POLYGON** (perpindahan polygon dengan gap tanpa observasi di pita), dan **ESTIMATED_TIMEOUT** (tambahan operasional otomatis setelah 60 frame sejak terakhir terlihat di polygon asal). Timeout tidak membuktikan pergantian ID atau crossing. Bukti baru dapat memperbarui satu passage tanpa tambahan +1 kedua; pembatalan harus tampil sebagai koreksi eksplisit.

Motorcycle+rider/passenger menggunakan satu unit tampilan hanya ketika relasinya didukung; BC counting tetap pengukuran bbox kendaraan. Kendaraan berbeda yang berdekatan tetap terpisah. ID track yang stabil merupakan asumsi identitas eksperimen, bukan ground truth.


In [ ]:
from phase_8_b_v2_batch import run_b_v2_batch

B_V2_REPORT = run_b_v2_batch(
    repo_root=REPO_ROOT, source_run_root=SOURCE_RUN_ROOT, source_video_root=SOURCE_VIDEO_ROOT,
    input_root=INPUT_ROOT, output_root=OUTPUT_ROOT, video_ids=VIDEO_IDS,
    render=RENDER, progress=print, run_id=RUN_ID,
)
print({key: B_V2_REPORT[key] for key in ('status', 'output_directory', 'selected_video_ids')
       if key in B_V2_REPORT})


## Review hasil

Buka tab **Output → results → phase_8_b_v2 → direktori run baru**. Simpan hasil dan manifest sebelum sesi berakhir. Notebook ini tidak mengunggah otomatis ke GitHub; publikasi terpisah harus mempertahankan checksum dan tautan audit.

- Pivot memisahkan kategori × arah; pending tidak masuk subtotal. A→B mendekati kamera dan B→A menjauhi kamera.
- **Log kendaraan lewat** berada di bawah passage accepted dan di atas pending. Baca ID/kategori, arah, jalur bukti, serta perubahan/koreksi event.
- Total operasional mencakup passage hasil inferensi polygon/timeout sesuai kebijakan pengguna; rincian terukur dan estimasi harus dibaca terpisah. `CHANGE_ID (ESTIMASI)` tidak membuktikan identitas benar-benar berubah.
- Warna bbox berbeda menurut kategori. Bbox gabungan komponen merupakan tampilan unit, tidak menggeser anchor BC sumber kendaraan.
- Timeline adalah replay ledger offline; posisi log pada frame tidak membuktikan seluruh informasi sudah tersedia secara kausal pada frame tersebut.
- Passage kembali yang didukung dapat dihitung lagi. Duplikasi ID pada satu passage tetap bukan passage baru.
- Bandingkan hasil B v1/B v2 per passage fisik: duplikasi, salah gabung, kategori, sumber crossing, kendaraan terlewat, dan salah tambah karena timeout. Penurunan pending saja belum membuktikan perbaikan.

Belum ada ground truth lengkap. Jangan melaporkan accuracy, precision, recall, F1, MAPE, tracking accuracy, atau counting accuracy dari jumlah accepted/pending. Jika sumber tidak tersedia di Codex, jalankan notebook pada Kaggle dengan input read-only yang disebutkan di atas.
